# MIMIC-CXR Image Downloader for 1,000 Patient Subset

This notebook matches the extracted 1,000 studies from `1mimic_subset_1000.csv` with the individual image paths listed in `IMAGE_FILENAMES`. It then downloads only those specific images from PhysioNet into your `restricted_Dowloaded_dataset` folder.

In [12]:
import os
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
from getpass import getpass
from tqdm.notebook import tqdm

## 1. Load the Subset and Filenames List
We load your cohort subset `1mimic_subset_1000.csv` and define the path to the `IMAGE_FILENAMES` manifest.

In [13]:
subset_path = '1mimic_subset_1000.csv'
filenames_path = 'IMAGE_FILENAMES'

df_subset = pd.read_csv(subset_path)
print(f"Loaded subset cohort with {len(df_subset)} studies.")

Loaded subset cohort with 1000 studies.


## 2. Match Study IDs with Filenames
We search the 377,111 entries in `IMAGE_FILENAMES` to find any chest X-ray image paths that match the `study_id`s in our 1,000-case subset.

In [14]:
# Store study IDs in a set for O(1) fast lookup
subset_study_ids = set(df_subset['study_id'].astype(str))

matched_image_paths = []

# Read IMAGE_FILENAMES line-by-line
with open(filenames_path, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # Path format is: files/p10/p10000032/s50414267/02aa804e-bde0afdd-112c0b34-7bc16630-4e384014.jpg
        parts = line.split('/')
        if len(parts) >= 4:
            study_part = parts[3]  # 's50414267'
            if study_part.startswith('s'):
                study_id = study_part[1:]  # '50414267'
                if study_id in subset_study_ids:
                    matched_image_paths.append(line)

print(f"Matched {len(matched_image_paths)} image files for the {len(subset_study_ids)} studies.")

Matched 1724 image files for the 1000 studies.


## 3. Create a Detailed Mapping File
We expand our subset table so that there is one row per individual image, matching each image path to its subject, study, and target labels. This is saved to `1mimic_subset_1000_images.csv`.

In [15]:
# Group matched paths by study_id
study_to_paths = {}
for path in matched_image_paths:
    sid = int(path.split('/')[3][1:])
    if sid not in study_to_paths:
        study_to_paths[sid] = []
    study_to_paths[sid].append(path)

# Expand rows (one row per image file)
expanded_rows = []
for _, row in df_subset.iterrows():
    sid = row['study_id']
    paths = study_to_paths.get(sid, [])
    for p in paths:
        expanded_rows.append({
            'subject_id': row['subject_id'],
            'study_id': row['study_id'],
            'Pneumonia': row['Pneumonia'],
            'No Finding': row['No Finding'],
            'image_path': p
        })

df_images = pd.DataFrame(expanded_rows)
df_images.to_csv('1mimic_subset_1000_images.csv', index=False)
print(f"Saved mapping list with {len(df_images)} rows to: 1mimic_subset_1000_images.csv")
df_images.head()

Saved mapping list with 1724 rows to: 1mimic_subset_1000_images.csv


,subject_id,study_id,Pneumonia,No Finding,image_path
0,19616701.0,50979586.0,NaN,1.0,files/p19/p19616701/s50979586/5d5e8c7a-24f6302...
1,19616701.0,50979586.0,NaN,1.0,files/p19/p19616701/s50979586/9d8da86b-a11600c...
2,16536183.0,52254730.0,0.0,1.0,files/p16/p16536183/s52254730/742327e4-d157057...
3,18504729.0,57450010.0,NaN,1.0,files/p18/p18504729/s57450010/549e1800-b0c3084...
4,18504729.0,57450010.0,NaN,1.0,files/p18/p18504729/s57450010/81805fab-16b732e...


## 4. Enter PhysioNet Credentials
Because the MIMIC-CXR dataset is restricted, you must input your PhysioNet username and password to download these images. 
> **Note:** The password field uses a secure mask so your credentials are not stored in the notebook.

In [17]:
download_dir = 'restricted_Dowloaded_dataset'
os.makedirs(download_dir, exist_ok=True)

print("Enter your PhysioNet login details:")
physionet_user = input("Username: ")
physionet_pass = getpass("Password: ")

Enter your PhysioNet login details:


Username:  ashutosh0001
Password:  ········


## 5. Download the Subset Images
We loop through our file list, recreate the folder structures in `restricted_Dowloaded_dataset`, and stream the images. It skips already downloaded files so you can stop and resume the download at any time.

**Note on headers:** We pass a standard browser `User-Agent` to prevent PhysioNet/Cloudflare from blocking the download.

In [18]:
# Version 2.1.0 is the active version on PhysioNet.
base_url = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/"
success_count = 0
skip_count = 0
error_count = 0

# Set a realistic browser User-Agent header so Cloudflare/PhysioNet doesn't block the request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

for path in tqdm(df_images['image_path'], desc="Downloading Images"):
    local_file_path = os.path.join(download_dir, path)
    
    # Skip already existing files
    if os.path.exists(local_file_path) and os.path.getsize(local_file_path) > 0:
        skip_count += 1
        continue
        
    # Create directories for the specific subject/study
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
    
    # Request file from PhysioNet
    url = base_url + path
    try:
        response = requests.get(
            url, 
            auth=HTTPBasicAuth(physionet_user, physionet_pass), 
            headers=headers, 
            stream=True
        )
        if response.status_code == 200:
            with open(local_file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            success_count += 1
        elif response.status_code == 403:
            print(f"\nHTTP 403 Forbidden: Access denied for '{path}'.")
            print("Your account might not have signed the DUA for this specific dataset on PhysioNet.")
            break
        elif response.status_code == 401:
            print(f"\nHTTP 401 Unauthorized: Invalid username or password.")
            break
        else:
            print(f"\nFailed to download: {path} (HTTP {response.status_code})")
            error_count += 1
    except Exception as e:
        print(f"\nError downloading {path}: {str(e)}")
        error_count += 1

print(f"\nDownload Summary:")
print(f"- Successfully downloaded: {success_count}")
print(f"- Skipped (already downloaded): {skip_count}")
print(f"- Failed/Errors: {error_count}")


HTTP 403 Forbidden: Access denied for 'files/p19/p19616701/s50979586/5d5e8c7a-24f63022-c201e4dd-23874daf-3a6c93f7.jpg'.
Your account might not have signed the DUA for this specific dataset on PhysioNet.

Download Summary:
- Successfully downloaded: 0
- Skipped (already downloaded): 0
- Failed/Errors: 0
